# Análisis de complejidad en señales EEG de sueño

**Dataset:** [Sleep-EDF Expanded (PhysioNet)](https://physionet.org/content/sleep-edfx/) - Kemp et al. 2000. Registros PSG de noche completa (EEG, EOG, EMG) con hipnogramas de etapas de sueño (W, N1, N2, N3, REM), uno de los datasets de referencia en la literatura de complejidad de EEG de sueño.

El objetivo es medir cómo cambia la complejidad de la señal EEG entre etapas de sueño, usando `mne` para la descarga y el preprocesamiento y `antropy` para las métricas de entropía / dimensión fractal.

## Flujo del análisis

La lógica reutilizable vive en el paquete [`src/`](../src); este notebook sólo la orquesta:

1. Descarga PSG + hipnograma de un sujeto/noche — `src.data.download_sleep_edf`.
2. Carga la señal y segmenta el EEG (canal Fpz-Cz) en épocas de 30 s etiquetadas por etapa de sueño (los estadios 3 y 4 se fusionan en N3) — `src.preprocessing`.
3. Calcula por época: entropía de permutación, entropía muestral, dimensión fractal de Higuchi y entropía espectral — `src.complexity`.
4. Promedia la complejidad por etapa (W, N1, N2, N3, REM). Típicamente se observa mayor complejidad en vigilia/REM y menor en sueño profundo (N3).
5. Exporta dos CSV a `data/processed/`.

In [1]:
%pip install -q mne antropy pandas matplotlib

import sys
from pathlib import Path

# Permite importar el paquete `src` estando el notebook en `notebooks/`.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.config import DATA_PROCESSED, EEG_CHANNEL
from src.data import download_sleep_edf
from src.preprocessing import load_recording, make_epochs
from src.complexity import complexity_dataframe, summarize_by_stage


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## 1. Descarga del dataset

Un sujeto y una noche, para mantenerlo liviano (hay 83 sujetos disponibles). Los archivos se guardan en `data/raw/` (si ya existen no se vuelven a descargar).

In [2]:
psg_path, hypnogram_path = download_sleep_edf(subject=0, recording=1)
print(f"PSG:        {psg_path}")
print(f"Hipnograma: {hypnogram_path}")

PSG:        C:\Users\matia\mne_data\physionet-sleep-data\SC4001E0-PSG.edf
Hipnograma: C:\Users\matia\mne_data\physionet-sleep-data\SC4001EC-Hypnogram.edf


## 2. Carga de la señal y segmentación en épocas

Se conserva el canal `EEG Fpz-Cz` y se lo segmenta en épocas de 30 s etiquetadas
por etapa de sueño (los estadios 3 y 4 se fusionan en N3).

In [ ]:
raw = load_recording(psg_path, hypnogram_path, channel=EEG_CHANNEL)
epochs = make_epochs(raw)

print(epochs)
print("\nÉpocas por etapa:")
print(pd.Series(epochs.events[:, -1]).map({v: k for k, v in epochs.event_id.items()}).value_counts())

## 3. Métricas de complejidad por época

Por cada época de 30 s se calcula: entropía de permutación, entropía muestral,
dimensión fractal de Higuchi y entropía espectral.

In [ ]:
df = complexity_dataframe(epochs)
df.head()

## 4. Resumen: complejidad promedio por etapa

Típicamente se observa mayor complejidad en vigilia/REM y menor en sueño profundo (N3).

In [ ]:
resumen = summarize_by_stage(df)
resumen

In [ ]:
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
por_epoca_csv = DATA_PROCESSED / "eeg_sleep_complexity_por_epoca.csv"
resumen_csv = DATA_PROCESSED / "eeg_sleep_complexity_resumen.csv"

df.to_csv(por_epoca_csv, index=False)
resumen.to_csv(resumen_csv)
print(f"Archivos guardados:\n  {por_epoca_csv}\n  {resumen_csv}")